## Report HTML

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np


# ─────────────────────────────────────────────────────────────────────────────
#  STYLE CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────

METHOD_LABELS: dict[str, str] = {
    "StandardLLR":                                                                       "Baseline",
    "FilteredLLR":                                                                       "Filtered",
    "theoretical_metrics":                                                               "Theoretical",
    "WeightedLLR - EntropyWeightingStrategy - normalize = True - operation = 1":        "Entropy (norm, ×)",
    "WeightedLLR - EntropyWeightingStrategy - normalize = True - operation = /":         "Entropy (norm, /)",
    "WeightedLLR - EntropyWeightingStrategy - normalize = False - operation = 1":       "Entropy (×)",
    "WeightedLLR - EntropyWeightingStrategy - normalize = False - operation = /":        "Entropy (/)",
    "WeightedLLR - KLWeightingStrategy - operation = /":                                 "KL (/)",
    "WeightedLLR - KLWeightingStrategy - operation = 1":                                 "KL (×)",
    "WeightedLLR - KLWeightingStrategy - operation = exp":                               "KL (exp)",
    "WeightedLLR - ParentsCountsWeightingStrategy - operation = /":              "ParCounts (/)",
    "WeightedLLR - ParentsCountsWeightingStrategy - operation = log":            "ParCounts (log)",
    "WeightedLLR - ParentsProbabilityWeightingStrategy - operation = /":                 "ParProb (/)",
    "WeightedLLR - ParentsProbabilityWeightingStrategy - operation = -":                 "ParProb (−)",
    "WeightedLLR - ParentsProbabilityWeightingStrategy - operation = -log":              "ParProb (−log)",
    "WeightedLLR - ParentsProbabilityWeightingStrategy - operation = 1":                 "ParProb (×)",
    "FilteredWeightedLLR - EntropyWeightingStrategy - normalize = True - operation = 1": "F+Entropy (norm, ×)",
    "FilteredWeightedLLR - EntropyWeightingStrategy - normalize = True - operation = /":  "F+Entropy (norm, /)",
    "FilteredWeightedLLR - EntropyWeightingStrategy - normalize = False - operation = 1": "F+Entropy (×)",
    "FilteredWeightedLLR - EntropyWeightingStrategy - normalize = False - operation = /":  "F+Entropy (/)",
    "FilteredWeightedLLR - KLWeightingStrategy - operation = /":                          "F+KL (/)",
    "FilteredWeightedLLR - KLWeightingStrategy - operation = 1":                          "F+KL (×)",
    "FilteredWeightedLLR - KLWeightingStrategy - operation = exp":                        "F+KL (exp)",
    "FilteredWeightedLLR - ParentsCountsWeightingStrategy - operation = /":      "F+ParCounts (/)",
    "FilteredWeightedLLR - ParentsCountsWeightingStrategy - operation = log":    "F+ParCounts (log)",
    "FilteredWeightedLLR - ParentsProbabilityWeightingStrategy - operation = /":          "F+ParProb (/)",
    "FilteredWeightedLLR - ParentsProbabilityWeightingStrategy - operation = -":          "F+ParProb (−)",
    "FilteredWeightedLLR - ParentsProbabilityWeightingStrategy - operation = -log":       "F+ParProb (−log)",
    "FilteredWeightedLLR - ParentsProbabilityWeightingStrategy - operation = 1":          "F+ParProb (×)",
}

SLOT_STYLES: list[dict] = [
    {"color": "#6b7280", "dash_css": "6,3",  "dash_plotly": "dot",   "width": 2.0},  # 0  Theoretical
    {"color": "#f59e0b", "dash_css": "none", "dash_plotly": "solid", "width": 2.5},  # 1  Baseline (amber)
    {"color": "#e63946", "dash_css": "none", "dash_plotly": "solid", "width": 2.5},  # 2  Filtered (red)
    {"color": "#3b82f6", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 3  blue
    {"color": "#10b981", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 4  emerald
    {"color": "#8b5cf6", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 5  purple
    {"color": "#f97316", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 6  orange
    {"color": "#06b6d4", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 7  cyan
    {"color": "#ec4899", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 8  pink
    {"color": "#84cc16", "dash_css": "none", "dash_plotly": "solid", "width": 1.8},  # 9  lime
    {"color": "#6b7280", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 10
    {"color": "#3b82f6", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 11
    {"color": "#10b981", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 12
    {"color": "#8b5cf6", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 13
    {"color": "#f97316", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 14
    {"color": "#06b6d4", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 15
    {"color": "#ec4899", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 16
    {"color": "#84cc16", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 17
    {"color": "#f59e0b", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 18
    {"color": "#e63946", "dash_css": "4,2",  "dash_plotly": "dash",  "width": 1.6},  # 19
    {"color": "#3b82f6", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 20
    {"color": "#10b981", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 21
    {"color": "#8b5cf6", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 22
    {"color": "#f97316", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 23
    {"color": "#06b6d4", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 24
    {"color": "#ec4899", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 25
    {"color": "#84cc16", "dash_css": "2,2",  "dash_plotly": "dot",   "width": 1.6},  # 26
]

MAX_ACTIVE = 6
DEFAULT_VISIBLE = {"theoretical_metrics", "StandardLLR", "FilteredLLR"}
CANONICAL_ORDER = ["theoretical_metrics", "StandardLLR", "FilteredLLR"]

def _round_list(lst, decimals=4):
    return [round(v, decimals) for v in lst]

# ─────────────────────────────────────────────────────────────────────────────
#  JSON PARSING
# ─────────────────────────────────────────────────────────────────────────────

def _load_json(path: Path) -> dict:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    return data[0] if isinstance(data, list) else data

def _extract_curves(data: dict) -> dict:
    curves: dict = {}
    for prior_entry in data["results"]:
        prior = prior_entry["prior_type"]
        curves[prior] = {}
        for coverage_entry in prior_entry["coverage_results"]:
            pop = float(coverage_entry["target_coverage"])
            curves[prior][pop] = {}
            for t_size_entry in coverage_entry["t_size_estimates"]:
                m = t_size_entry["target_size_multiplier"]
                curves[prior][pop][m] = {}
                for bn_result in t_size_entry["bn_results"]:
                    for method_key, metric in bn_result["avg_metrics"].items():
                        if method_key not in curves[prior][pop][m]:
                            curves[prior][pop][m][method_key] = {"fpr_list": [], "tpr_list": []}
                        curves[prior][pop][m][method_key]["fpr_list"].append(metric["fpr"])
                        curves[prior][pop][m][method_key]["tpr_list"].append(metric["tpr"])
    return curves

def _median_iqr(arrays: list) -> tuple:
    mat = np.array(arrays)
    return (
        np.median(mat, axis=0),
        np.percentile(mat, 25, axis=0),
        np.percentile(mat, 75, axis=0),
    )


def _pauc(fpr: np.ndarray, tpr: np.ndarray, fpr_max: float = 0.2) -> float:
    mask = fpr <= fpr_max
    if mask.sum() < 2:
        return float("nan")
    if fpr[mask][-1] < fpr_max and mask.sum() < len(fpr):
        idx = int(np.where(~mask)[0][0])
        t = (fpr_max - fpr[idx - 1]) / (fpr[idx] - fpr[idx - 1])
        tpr_interp = tpr[idx - 1] + t * (tpr[idx] - tpr[idx - 1])
        f = np.append(fpr[mask], fpr_max)
        t_ = np.append(tpr[mask], tpr_interp)
    else:
        f, t_ = fpr[mask], tpr[mask]
    return float(np.trapezoid(t_, f) / fpr_max)


def _extract_network_info(data: dict) -> dict:
    cfg = data.get("config", {})
    bn_cfg = cfg.get("bn", {})
    exp_cfg = cfg.get("exp_settings", {})
    bns = data.get("bns", [])

    n_bn_extractions = exp_cfg.get("n_bn_extractions")
    is_synthetic = bool(n_bn_extractions) and len(bns) > 1

    ratio_arc = bn_cfg.get("ratio_arc")

    info: dict = {
        "n_nodes": bn_cfg.get("n_nodes"),
        "ratio_arc": round(ratio_arc, 2) if ratio_arc is not None else None,
        "n_modmin": bn_cfg.get("n_modmin"),
        "n_modmax": bn_cfg.get("n_modmax"),
        "prior_type_vec": exp_cfg.get("prior_type_vec", []),
        "n_sample_extractions": exp_cfg.get("n_sample_extractions"),
        "n_realizations": len(bns),
        "is_synthetic": is_synthetic,
    }

    complexities = [b["bn_complexity"] for b in bns if "bn_complexity" in b]
    if complexities:
        if len(complexities) > 1:
            arr = np.array(complexities, dtype=float)
            info["complexity_median"] = float(np.median(arr))
            info["complexity_q1"] = float(np.percentile(arr, 25))
            info["complexity_q3"] = float(np.percentile(arr, 75))
        else:
            info["complexity"] = float(complexities[0])

    first_prior = data["results"][0]
    first_cov = first_prior["coverage_results"][0]
    multipliers = [e["target_size_multiplier"] for e in first_cov["t_size_estimates"]]
    info["multipliers"] = multipliers

    return info


# ─────────────────────────────────────────────────────────────────────────────
#  FOLDER STRUCTURE DISCOVERY
# ─────────────────────────────────────────────────────────────────────────────

def _natural_sort_key(s: str) -> list:
    return [int(c) if c.isdigit() else c.lower() for c in re.split(r"(\d+)", s)]


def _find_leaves(root: Path) -> list[dict]:
    leaves = []
    for p in sorted(root.rglob("experiment_results.json"),
                    key=lambda x: _natural_sort_key(str(x))):
        rel = p.parent.relative_to(root)
        parts = list(rel.parts)
        leaves.append({
            "path": p,
            "parts": parts,
            "label": parts[-1] if parts else str(p),
        })
    return leaves


def _build_tree(leaves: list[dict]) -> dict:
    root: dict = {"__children__": {}, "__leaf__": None}
    for leaf in leaves:
        node = root
        for part in leaf["parts"]:
            node["__children__"].setdefault(part, {"__children__": {}, "__leaf__": None})
            node = node["__children__"][part]
        node["__leaf__"] = leaf
    return root


# ─────────────────────────────────────────────────────────────────────────────
#  PLOTLY DATA GENERATION
# ─────────────────────────────────────────────────────────────────────────────

def _build_plot_data(json_path: Path, fpr_max: float, show_iqr: bool) -> dict:
    data = _load_json(json_path)
    curves = _extract_curves(data)
    network_info = _extract_network_info(data)

    prior_types = list(curves.keys())
    pop_sizes = sorted(curves[prior_types[0]].keys())

    all_methods: set[str] = set()
    for pop in pop_sizes:
        for prior in prior_types:
            for m_entry in curves[prior][pop].values():
                all_methods |= set(m_entry.keys())

    ordered_methods = []
    for m in CANONICAL_ORDER:
        if m in all_methods:
            ordered_methods.append(m)
    for m in sorted(all_methods - set(CANONICAL_ORDER), key=_natural_sort_key):
        ordered_methods.append(m)

    n_styles = len(SLOT_STYLES)
    slot_map: dict[str, int] = {}
    fixed = {"theoretical_metrics": 0, "StandardLLR": 1, "FilteredLLR": 2}
    next_slot = 3
    for m in ordered_methods:
        if m in fixed:
            slot_map[m] = fixed[m]
        else:
            slot_map[m] = next_slot % n_styles
            next_slot += 1

    plottable = ordered_methods
    traces = []

    multipliers = network_info["multipliers"]

    for prior_type in prior_types:
        for pop_idx, pop in enumerate(pop_sizes):
            for m in multipliers:
                for method_key in plottable:
                    if method_key not in curves[prior_type][pop][m]:
                        continue

                    vals = curves[prior_type][pop][m][method_key]
                    slot = slot_map[method_key]
                    style = SLOT_STYLES[slot]
                    label = METHOD_LABELS.get(method_key, method_key)
                    visible = method_key in DEFAULT_VISIBLE

                    n = len(vals["fpr_list"])
                    show_legend = True

                    if n == 1:
                        fpr = np.array(vals["fpr_list"][0])
                        tpr = np.array(vals["tpr_list"][0])
                        mask = fpr <= fpr_max + 1e-9
                        traces.append({
                            "x": _round_list(fpr[mask].tolist()),
                            "y": _round_list(tpr[mask].tolist()),
                            "type": "scatter",
                            "mode": "lines",
                            "name": label,
                            "legendgroup": method_key,
                            "showlegend": show_legend,
                            "visible": visible,
                            "line": {"color": style["color"], "width": style["width"], "dash": style["dash_plotly"]},
                            "hovertemplate": f"{label} | cov={pop}<br>FPR=%{{x:.3f}} TPR=%{{y:.3f}}<extra></extra>",
                            "meta": {"method_key": method_key, "pop": pop, "multiplier": m, "prior": prior_type, "is_iqr": False},
                        })
                    else:
                        med_fpr, _, _ = _median_iqr(vals["fpr_list"])
                        med_tpr, q1_tpr, q3_tpr = _median_iqr(vals["tpr_list"])
                        mask = med_fpr <= fpr_max + 1e-9
                        pauc_val = _pauc(med_fpr, med_tpr, fpr_max=min(fpr_max, 0.2))
                        pauc_str = f"{pauc_val:.3f}" if not np.isnan(pauc_val) else "n/a"

                        if show_iqr:
                            x_band = np.concatenate([med_fpr[mask], med_fpr[mask][::-1]])
                            y_band = np.concatenate([q3_tpr[mask], q1_tpr[mask][::-1]])
                            traces.append({
                                "x": _round_list(x_band.tolist()),
                                "y": _round_list(y_band.tolist()),
                                "type": "scatter",
                                "mode": "lines",
                                "fill": "toself",
                                "fillcolor": style["color"],
                                "opacity": 0.12,
                                "line": {"width": 0},
                                "name": label + " IQR",
                                "legendgroup": method_key,
                                "showlegend": False,
                                "visible": visible,
                                "hoverinfo": "skip",
                                "meta": {"method_key": method_key, "pop": pop, "multiplier": m, "prior": prior_type, "is_iqr": True},
                            })

                        traces.append({
                            "x": _round_list(med_fpr[mask].tolist()),
                            "y": _round_list(med_tpr[mask].tolist()),
                            "type": "scatter",
                            "mode": "lines",
                            "name": label,
                            "legendgroup": method_key,
                            "showlegend": show_legend,
                            "visible": visible,
                            "line": {"color": style["color"], "width": style["width"], "dash": style["dash_plotly"]},
                            "hovertemplate": f"{label} | cov={pop}<br>FPR=%{{x:.3f}} TPR=%{{y:.3f}}<br>pAUC(0.2)={pauc_str}<extra></extra>",
                            "meta": {"method_key": method_key, "pop": pop, "multiplier": m, "prior": prior_type, "is_iqr": False},
                        })
                        
        diag = _round_list(np.linspace(0, fpr_max, 60).tolist())
        for m in multipliers:
            for pop in pop_sizes:
                traces.append({
                    "x": diag, "y": diag,
                    "type": "scatter", "mode": "lines",
                    "line": {"color": "#cccccc", "dash": "dot", "width": 1},
                    "showlegend": False, "visible": True, "hoverinfo": "skip",
                    "meta": {"method_key": "__diag__", "pop": pop, "multiplier": m, "prior": prior_type, "is_iqr": False},
                })

    methods_info = []
    for m in plottable:
        slot = slot_map[m]
        style = SLOT_STYLES[slot]
        methods_info.append({
            "key": m,
            "label": METHOD_LABELS.get(m, m),
            "slot": slot,
            "color": style["color"],
            "dash_css": style["dash_css"],
            "default_visible": m in DEFAULT_VISIBLE,
        })

    return {
        "traces": traces,
        "methods": methods_info,
        "coverage_targets": [float(p) for p in pop_sizes],
        "prior_types": prior_types,
        "fpr_max": fpr_max,
        "network": network_info,
        "multipliers": network_info["multipliers"],
    }

# ─────────────────────────────────────────────────────────────────────────────
#  HTML SIDEBAR TREE RENDERING
# ─────────────────────────────────────────────────────────────────────────────

def _render_tree_html(node: dict, leaf_index: list[int], depth: int = 0) -> str:
    html_parts = []
    BASE_INDENT = 10
    GROUP_BASE  = 10

    for name, child in sorted(node["__children__"].items(),
                               key=lambda x: _natural_sort_key(x[0])):
        is_leaf = child["__leaf__"] is not None and not child["__children__"]

        if is_leaf:
            idx = leaf_index[0]
            leaf_index[0] += 1
            is_default = idx == 0
            leaf_indent = GROUP_BASE + (depth + 1) * BASE_INDENT + 6
            html_parts.append(
                f'<div class="tree-leaf{" active" if is_default else ""}" '
                f'data-idx="{idx}" onclick="loadPlot({idx})" '
                f'style="padding-left:{leaf_indent}px">'
                f'<span class="leaf-icon">◆</span> {name}</div>'
            )
        else:
            children_html = _render_tree_html(child, leaf_index, depth + 1)
            open_attr = ""
            group_indent = GROUP_BASE + depth * BASE_INDENT
            html_parts.append(
                f'<details class="tree-details" {open_attr} data-default-open="{"true" if depth < 2 else "false"}">'
                f'<summary class="tree-group" style="padding-left:{group_indent}px">'
                f'<span class="group-icon">▸</span> {name}</summary>'
                f'<div class="tree-children">{children_html}</div>'
                f'</details>'
            )

    return "".join(html_parts)


# ─────────────────────────────────────────────────────────────────────────────
#  MAIN FUNCTION
# ─────────────────────────────────────────────────────────────────────────────

def build_roc_report(
    root: str | Path,
    output: str | Path = "roc_report.html",
    fpr_max: float = 0.3,
    show_iqr: bool = True,
    title: str = "Experiment Browser",
) -> Path:
    root = Path(root)
    output = Path(output)

    print(f"📂  Scanning {root} …")
    leaves = _find_leaves(root)

    if not leaves:
        raise FileNotFoundError(
            f"No experiment_results.json file found in {root}"
        )

    print(f"✅  {len(leaves)} experiments found")

    all_plot_data = []
    for i, leaf in enumerate(leaves):
        print(f"   [{i+1:3d}/{len(leaves)}]  {'/'.join(leaf['parts'])}", end="\r")
        try:
            pd = _build_plot_data(leaf["path"], fpr_max=fpr_max, show_iqr=show_iqr)
            pd["breadcrumb"] = leaf["parts"]
            pd["title"] = " › ".join(leaf["parts"])
            all_plot_data.append(pd)
        except Exception as e:
            print(f"\n   ⚠️  Error on {leaf['path']}: {e}")
            all_plot_data.append(None)

    print(f"\n🎨  Generating HTML …")

    tree = _build_tree(leaves)
    leaf_index = [0]
    sidebar_html = _render_tree_html(tree, leaf_index)

    plots_json = json.dumps(all_plot_data, allow_nan=False)

    html = _render_html(title, sidebar_html, plots_json, fpr_max)
    output.write_text(html, encoding="utf-8")

    size_kb = output.stat().st_size / 1024
    print(f"✅  Report saved to: {output}  ({size_kb:.0f} KB)")
    return output


# ─────────────────────────────────────────────────────────────────────────────
#  HTML TEMPLATE
# ─────────────────────────────────────────────────────────────────────────────

def _render_html(title: str, sidebar_html: str, plots_json: str, fpr_max: float) -> str:
    return f"""<!doctype html>
<html lang="en" data-theme="dark">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{title}</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&family=JetBrains+Mono:wght@400;500&display=swap">
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js" charset="utf-8"></script>
<style>
/* ── Reset & base ── */
*, *::before, *::after {{ box-sizing: border-box; margin: 0; padding: 0; }}

:root {{
  --font:      'Inter', system-ui, sans-serif;
  --mono:      'JetBrains Mono', monospace;
  --sidebar-w: 280px;
  --header-h:  50px;
  --radius:    8px;
  --accent:    #3b82f6;
  --ctrl-h:    auto;
}}

/* ── Dark theme (default) ── */
:root, [data-theme="dark"] {{
  --bg:          #0f1117;
  --bg2:         #181c27;
  --bg3:         #1e2336;
  --border:      #2a3050;
  --text:        #e2e8f0;
  --text2:       #94a3b8;
  --text3:       #64748b;
  --leaf-hover:  #252d44;
  --leaf-active: #1e3a5f;
  --plot-bg:     #181c27;
  --plot-grid:   #2a3050;
  --plot-paper:  rgba(0,0,0,0);
}}

/* ── Light theme ── */
[data-theme="light"] {{
  --bg:          #f1f5f9;
  --bg2:         #ffffff;
  --bg3:         #e8edf4;
  --border:      #c8d3e0;
  --text:        #0f172a;
  --text2:       #1e293b;
  --text3:       #475569;
  --leaf-hover:  #dbeafe;
  --leaf-active: #bfdbfe;
  --plot-bg:     #ffffff;
  --plot-grid:   #cbd5e1;
  --plot-paper:  rgba(0,0,0,0);
}}

html, body {{
  height: 100%;
  background: var(--bg);
  color: var(--text);
  font-family: var(--font);
  font-size: 14px;
  line-height: 1.5;
  overflow: hidden;
}}

/* ── Layout shell ── */
#app {{
  display: grid;
  grid-template-rows: var(--header-h) 1fr;
  grid-template-columns: var(--sidebar-w) 4px 1fr;
  height: 100vh;
}}

/* ── Header ── */
#header {{
  grid-column: 1 / -1;
  display: flex;
  align-items: center;
  gap: 12px;
  padding: 0 20px;
  background: var(--bg2);
  border-bottom: 1px solid var(--border);
  user-select: none;
}}
#header .logo {{
  font-size: 17px;
  font-weight: 600;
  letter-spacing: -0.3px;
  color: var(--text);
}}
#header .logo span {{ color: var(--accent); }}
#header .subtitle {{
  font-size: 12px;
  color: var(--text3);
  margin-left: 2px;
}}
#header .spacer {{ flex: 1; }}
#header .badge {{
  font-size: 11px;
  font-family: var(--mono);
  background: var(--bg3);
  border: 1px solid var(--border);
  border-radius: 4px;
  padding: 2px 8px;
  color: var(--text2);
}}

/* ── Sidebar ── */
#sidebar {{
  background: var(--bg2);
  border-right: 1px solid var(--border);
  overflow-y: auto;
  overflow-x: hidden;
  scrollbar-width: thin;
  scrollbar-color: var(--border) transparent;
  display: flex;
  flex-direction: column;
  min-width: 160px;
  max-width: 520px;
  position: relative;
  transition: width 0.2s ease;
}}
#sidebar.collapsed {{
  min-width: 0 !important;
  width: 0 !important;
  overflow: hidden;
}}
#sidebar::-webkit-scrollbar {{ width: 4px; }}
#sidebar::-webkit-scrollbar-thumb {{ background: var(--border); border-radius: 2px; }}

/* Sidebar collapse toggle (inside sidebar header) */
#sidebar-header {{
  display: flex;
  align-items: center;
  gap: 6px;
  padding: 8px 10px 0 12px;
  flex-shrink: 0;
}}
#sidebar-title {{
  font-size: 11px;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.6px;
  color: var(--text3);
  flex: 1;
}}
#collapse-btn {{
  background: none;
  border: 1px solid var(--border);
  border-radius: 5px;
  color: var(--text3);
  cursor: pointer;
  font-size: 13px;
  padding: 2px 6px;
  line-height: 1;
  transition: all .15s;
  flex-shrink: 0;
}}
#collapse-btn:hover {{
  background: var(--accent);
  color: #fff;
  border-color: var(--accent);
}}

/* Sidebar resize handle */
#resize-handle {{
  background: transparent;
  cursor: col-resize;
  position: relative;
  z-index: 20;
  transition: background 0.15s;
  flex-shrink: 0;
}}
#resize-handle:hover, #resize-handle.dragging {{
  background: var(--accent);
  opacity: 0.5;
}}
/* Sidebar expand button (visible only when collapsed) */

#expand-btn {{
  display: block;
  position: fixed;
  top: calc(var(--header-h) + 40px);
  left: 0;
  z-index: 30;
  background: var(--bg2);
  border: 1px solid var(--border);
  border-left: none;
  border-radius: 0 6px 6px 0;
  color: transparent;
  cursor: pointer;
  font-size: 14px;
  padding: 20px 3px;
  width: 8px;
  overflow: hidden;
  opacity: 0;
  pointer-events: none;
  transition: width 0.2s ease, padding 0.2s ease, opacity 0.15s ease;
  box-shadow: 2px 0 6px rgba(0,0,0,0.2);
}}
#expand-btn.visible {{
  opacity: 1;
  pointer-events: auto;
}}
#expand-btn:hover {{
  width: 28px;
  padding: 20px 7px;
  background: var(--accent);
  color: #fff;
  border-color: var(--accent);
}}

#search-wrap {{
  padding: 10px 10px 8px;
  position: sticky;
  top: 0;
  background: var(--bg2);
  border-bottom: 1px solid var(--border);
  z-index: 10;
  position: relative;
}}
#search {{
  width: 100%;
  background: var(--bg3);
  border: 1px solid var(--border);
  border-radius: var(--radius);
  color: var(--text);
  font-family: var(--font);
  font-size: 13px;
  padding: 5px 10px 5px 28px;
  outline: none;
  transition: border-color .15s;
}}
#search::placeholder {{ color: var(--text3); }}
#search:focus {{ border-color: var(--accent); }}
#search-icon {{
  position: absolute;
  left: 20px;
  top: 50%;
  transform: translateY(-50%);
  color: var(--text3);
  font-size: 11px;
  pointer-events: none;
}}

#tree {{ padding: 6px 0 24px; flex: 1; }}

/* Tree nodes */
.tree-details {{ border: none; }}
.tree-details > summary {{ list-style: none; cursor: pointer; }}
.tree-details > summary::-webkit-details-marker {{ display: none; }}

.tree-group {{
  display: flex;
  align-items: center;
  gap: 6px;
  padding: 5px 14px;
  font-size: 11px;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.4px;
  color: var(--text3);
  cursor: pointer;
  transition: color .1s;
  user-select: none;
}}
.tree-group:hover {{ color: var(--text2); }}
.group-icon {{
  font-size: 9px;
  transition: transform .15s;
  display: inline-block;
}}
details[open] > summary .group-icon {{ transform: rotate(90deg); }}
.tree-children {{ padding-left: 0; }}

.tree-leaf {{
  display: flex;
  align-items: center;
  gap: 7px;
  padding: 5px 14px;
  font-size: 13px;
  color: var(--text2);
  cursor: pointer;
  border-left: 2px solid transparent;
  transition: background .1s, color .1s, border-color .1s;
  user-select: none;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}}
.tree-leaf:hover {{ background: var(--leaf-hover); color: var(--text); }}
.tree-leaf.active {{
  background: var(--leaf-active);
  color: var(--accent);
  border-left-color: var(--accent);
}}
.leaf-icon {{ font-size: 7px; color: var(--text3); flex-shrink: 0; }}
.tree-leaf.active .leaf-icon {{ color: var(--accent); }}
.tree-leaf.hidden {{ display: none; }}

/* ── Main panel ── */
#main {{
  display: flex;
  flex-direction: column;
  overflow: hidden;
  background: var(--bg);
  min-width: 0;
}}

/* Breadcrumb */
#breadcrumb {{
  display: flex;
  align-items: center;
  gap: 4px;
  padding: 8px 18px;
  border-bottom: 1px solid var(--border);
  font-size: 13px;
  flex-shrink: 0;
  flex-wrap: wrap;
  background: var(--bg2);
}}
.bc-item {{ color: var(--text3); font-weight: 500; padding: 2px 4px; border-radius: 4px; }}
.bc-sep {{ color: var(--text3); font-weight: 700; opacity: 0.6; padding: 0 1px; }}
.bc-item:last-child {{ color: var(--accent); font-weight: 600; background: var(--bg3); }}

/* Network info bar */
#network-info {{
  display: flex;
  align-items: center;
  gap: 6px;
  padding: 6px 18px;
  border-bottom: 1px solid var(--border);
  background: var(--bg2);
  flex-shrink: 0;
  flex-wrap: wrap;
}}
.net-badge {{
  display: inline-flex;
  align-items: center;
  gap: 5px;
  font-size: 12px;
  background: var(--bg3);
  border: 1px solid var(--border);
  border-radius: 6px;
  padding: 2px 8px;
  color: var(--text);
  white-space: nowrap;
}}
.net-badge .net-key {{
  color: var(--text3);
  font-family: var(--font);
  font-size: 11px;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.3px;
}}
.net-badge .net-val {{
  color: var(--text);
  font-family: var(--mono);
  font-size: 12px;
  font-weight: 500;
}}

#controls-panel {{
  background: var(--bg2);
  border-bottom: 1px solid var(--border);
  flex-shrink: 0;
  overflow: hidden;
  transition: max-height 0.25s ease, padding 0.25s ease;
  max-height: 300px;
}}
#controls-panel.collapsed {{
  max-height: 0;
  border-bottom-color: transparent;
}}

#controls-toggle {{
  display: flex;
  align-items: center;
  gap: 8px;
  padding: 5px 18px;
  cursor: pointer;
  background: var(--bg2);
  border-bottom: 1px solid var(--border);
  user-select: none;
  flex-shrink: 0;
}}
#controls-toggle:hover {{ background: var(--bg3); }}
#controls-toggle-label {{
  font-size: 11px;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.5px;
  color: var(--text3);
  flex: 1;
}}
#controls-toggle-arrow {{
  font-size: 10px;
  color: var(--text3);
  transition: transform 0.2s;
}}
#controls-toggle-arrow.open {{ transform: rotate(180deg); }}

/* Inner controls layout: two rows */
#controls-inner {{
  display: flex;
  flex-direction: column;
  gap: 0;
  padding: 8px 18px 10px;
}}

/* Row 1: Coverage target */
.ctrl-row {{
  display: flex;
  align-items: center;
  gap: 16px;
  padding: 4px 0;
  flex-wrap: wrap;
}}
.ctrl-label {{
  font-size: 11px;
  font-weight: 600;
  margin-right: 8px;
  text-transform: uppercase;
  letter-spacing: 0.5px;
  color: var(--text3);
  white-space: nowrap;
  min-width: 100px;
}}

/* Row 2: Curve buttons — wrap freely across full width */
#curve-row {{
  display: flex;
  align-items: flex-start;
  gap: 8px;
  padding: 4px 0;
  flex-wrap: wrap;
}}
#curve-label {{
  font-size: 11px;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.5px;
  color: var(--text3);
  white-space: nowrap;
  min-width: 100px;
  padding-top: 5px;
}}
#curve-buttons {{
  display: grid;
  grid-template-columns: repeat(auto-fill, minmax(180px, 1fr)); 
  gap: 8px 12px;
  flex: 1;
  width: 100%;
}}
#curve-count {{
  display: none !important; 
}}

.curve-btn {{
  display: inline-flex;
  align-items: center;
  gap: 5px;
  padding: 5px 12px;
  border-radius: 20px;
  border: 1.5px solid transparent;
  font-family: var(--font);
  font-size: 12px;
  font-weight: 500;
  cursor: pointer;
  transition: all .15s;
  background: var(--bg3);
  color: var(--text2);
  white-space: nowrap;
  justify-content: flex-start;
}}
.curve-btn .swatch {{
  width: 16px;
  height: 4px;
  border-radius: 2px;
  flex-shrink: 0;
}}
.curve-btn.active {{
  color: var(--text);
  border-color: currentColor;
  background: transparent;
}}
.curve-btn:hover:not(.active):not(.disabled) {{
  background: var(--bg3);
  border-color: var(--border);
  color: var(--text);
}}
.curve-btn.disabled {{
  opacity: 0.3;
  cursor: not-allowed;
}}

/* Pop size buttons */
.pop-btn {{
  padding: 3px 10px;
  border-radius: 5px;
  border: 1px solid var(--border);
  background: var(--bg3);
  color: var(--text2);
  font-family: var(--mono);
  font-size: 12px;
  cursor: pointer;
  transition: all .12s;
}}
.pop-btn.active {{
  background: var(--accent);
  border-color: var(--accent);
  color: #fff;
}}
.pop-btn:hover:not(.active) {{ border-color: var(--accent); color: var(--text); }}

/* ── Plot area ── */
#plot-container {{
  flex: 1;
  overflow: hidden;
  position: relative;
  min-height: 0;
}}
#plot {{ width: 100%; height: 100%; }}

/* Empty state */
#empty-state {{
  display: none;
  flex-direction: column;
  align-items: center;
  justify-content: center;
  height: 100%;
  gap: 12px;
  color: var(--text3);
}}
#empty-state .big {{ font-size: 48px; }}
#empty-state p {{ font-size: 14px; }}

/* Loading overlay */
#loading {{
  display: none;
  position: absolute;
  inset: 0;
  align-items: center;
  justify-content: center;
  background: rgba(15,17,23,0.7);
  backdrop-filter: blur(2px);
  z-index: 100;
  font-size: 13px;
  color: var(--text2);
  gap: 10px;
}}
#loading.visible {{ display: flex; }}
.spinner {{
  width: 18px; height: 18px;
  border: 2px solid var(--border);
  border-top-color: var(--accent);
  border-radius: 50%;
  animation: spin .7s linear infinite;
}}
@keyframes spin {{ to {{ transform: rotate(360deg); }} }}

/* Theme toggle btn */
#theme-btn {{
  background: var(--bg3);
  border: 1px solid var(--border);
  border-radius: 6px;
  color: var(--text2);
  cursor: pointer;
  font-size: 15px;
  padding: 3px 8px;
  line-height: 1;
  transition: all .15s;
}}
#theme-btn:hover {{ background: var(--border); color: var(--text); }}
#plot-legend {{
  display: none;
  flex-wrap: wrap;
  gap: 6px 16px;
  padding: 6px 18px 8px;
  background: var(--bg2);
  border-top: 1px solid var(--border);
  flex-shrink: 0;
}}
#plot-legend.visible {{ display: flex; justify-content: center; flex-wrap: wrap; }}
.legend-item {{
  display: flex;
  justify-content: center;
  align-items: center;
  gap: 6px;
  font-size: 12px;
  color: var(--text2);
}}

#pop-buttons {{
  display: flex;
  gap: 6px;
  flex-wrap: wrap;
}}
.modebar-container {{
  display: none !important;
}}
</style>
</head>
<body>
<div id="app">

  <!-- Header -->
  <header id="header">
    <div class="logo">Experiment <span>Browser</span></div>
    <div class="spacer"></div>
    <button id="theme-btn" onclick="toggleTheme()" title="Toggle Light/Dark Theme">🌗</button>
  </header>

  <!-- Sidebar -->
  <aside id="sidebar">
    <div id="sidebar-header">
      <div id="sidebar-title">Experiments</div>
      <button id="collapse-btn" onclick="toggleSidebar()" title="Collapse sidebar">◀</button>
    </div>
    <div id="search-wrap">
      <span id="search-icon">🔍</span>
      <input type="text" id="search" placeholder="Search experiments..." oninput="onSearch()">
    </div>
    <div id="tree">
      {sidebar_html}
    </div>
  </aside>

  <!-- Sidebar Resize Handle -->
  <div id="resize-handle"></div>

  <!-- Expand button (floats over, hidden by default) -->
  <button id="expand-btn" onclick="toggleSidebar()">▶</button>

  <!-- Main Panel -->
  <main id="main">
    <div id="breadcrumb"></div>
    <div id="network-info"></div>

    <!-- Controls Panel -->
    <div id="controls-toggle" onclick="toggleControls()">
      <span id="controls-toggle-label">Curves</span>
      <span id="controls-toggle-arrow" class="open">▼</span>
    </div>
    
    <div id="controls-panel">
      <div id="controls-inner">
        <!-- Curves Legend Row -->
        <div id="curve-row">
          <div id="curve-buttons"></div>
        </div>
      </div>
    </div>
    
    <div id="pop-row" style="display:flex;align-items:center;gap:16px;padding:6px 18px;background:var(--bg2);border-bottom:1px solid var(--border);flex-shrink:0;flex-wrap:wrap;">
      <div class="ctrl-label">Coverage Target</div>
      <div id="pop-buttons"></div>
      <div id="viewmode-toggle" onclick="toggleViewMode()" title="Switch view mode" style="display:flex;align-items:center;gap:8px;cursor:pointer;margin-left:auto;">
        <span id="label-multicov" style="font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;color:var(--text3);">Multi Coverage</span>
        <div style="position:relative;width:44px;height:24px;background:var(--accent);border-radius:12px;transition:background .3s;flex-shrink:0;" id="toggle-track">
          <div id="toggle-thumb" style="position:absolute;top:3px;left:3px;width:18px;height:18px;background:#fff;border-radius:50%;transition:transform .3s;transform:translateX(20px);"></div>
        </div>
        <span id="label-multimul" style="font-size:11px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;color:var(--accent);">Multi Multiplier</span>
      </div>
      <div id="mul-label" class="ctrl-label" style="margin-left:16px; min-width:unset; display:none;">Multiplier</div>
      <div id="mul-buttons" style="display:flex;gap:6px;flex-wrap:wrap;"></div>
    </div>

    <!-- Plot Area -->
    <div id="plot-container">
      <div id="loading"><div class="spinner"></div>Loading dataset...</div>
      <div id="empty-state">
        <span class="big">📈</span>
        <p>Select an experiment from the sidebar to view curves.</p>
      </div>
      <div id="plot"></div>
    </div>
    <div id="plot-legend"></div>
  </main>


</div>

<script>
// --- JS STATE & LOGIC ---
const plotsData = {plots_json};
let currentIdx = 0;
let selectedPop = null;
let viewMode = "multi_mul"; // "multi_mul" | "multi_cov"
let selectedPrior = null;
let activeMethods = new Set();
let theme = "dark";

const plotDiv = document.getElementById('plot');
const loading = document.getElementById('loading');
const emptyState = document.getElementById('empty-state');

window.addEventListener('DOMContentLoaded', () => {{
    initResize();
    if (plotsData.length > 0) {{
        loadPlot(0);
        expandToLeaf(0);
    }} else {{
        showEmptyState();
    }}
    setTimeout(repositionExpandBtn, 100);
}});

function toggleTheme() {{
    theme = theme === "dark" ? "light" : "dark";
    document.documentElement.setAttribute('data-theme', theme);
    if (plotDiv.data) {{
        updatePlotColors();
    }}
    const modebar = plotDiv.querySelector('.modebar-container');
    if (modebar) modebar.style.background = theme === 'light' ? 'transparent' : '';
}}

function updatePlotColors() {{
    const isDark = theme === "dark";
    const bg = isDark ? "#181c27" : "#ffffff";
    const grid = isDark ? "#2a3050" : "#cbd5e1";
    const text = isDark ? "#e2e8f0" : "#0f172a";
    
    const layoutUpdate = {{
        plot_bgcolor: bg,
        paper_bgcolor: "rgba(0,0,0,0)",
        font: {{ color: text, family: "'Inter', sans-serif" }},
        'xaxis.gridcolor': grid,
        'yaxis.gridcolor': grid,
        'xaxis.linecolor': grid,
        'yaxis.linecolor': grid
    }};
    
    for (let i = 1; i <= 10; i++) {{
        layoutUpdate[`xaxis${{i}}.gridcolor`] = grid;
        layoutUpdate[`yaxis${{i}}.gridcolor`] = grid;
        layoutUpdate[`xaxis${{i}}.linecolor`] = grid;
        layoutUpdate[`yaxis${{i}}.linecolor`] = grid;
    }}
    
    if (plotDiv.layout && plotDiv.layout.annotations) {{
        layoutUpdate.annotations = plotDiv.layout.annotations.map(a => ({{
            ...a, font: {{ ...a.font, color: text }}
        }}));
    }}
    
    Plotly.relayout(plotDiv, layoutUpdate);
}}

function loadPlot(idx) {{
    currentIdx = idx;
    const data = plotsData[idx];
    if (!data) {{ showEmptyState(); return; }}

    if (selectedPop === null || !data.coverage_targets.includes(selectedPop)) {{
        selectedPop = data.coverage_targets[0];
    }}
    
    if (selectedPrior === null || !data.prior_types.includes(selectedPrior)) {{
        selectedPrior = data.prior_types[0];
    }}

    if (activeMethods.size === 0) {{
        activeMethods = new Set(data.methods.filter(m => m.default_visible).map(m => m.key));
    }}

    // Highlight sidebar
    document.querySelectorAll('.tree-leaf').forEach(el => el.classList.remove('active'));
    const activeLeaf = document.querySelector(`.tree-leaf[data-idx="${{idx}}"]`);
    
    if (activeLeaf) activeLeaf.classList.add('active');

    emptyState.style.display = "none";
    loading.classList.add('visible');

    // Breadcrumb
    const bc = document.getElementById('breadcrumb');
    bc.innerHTML = data.breadcrumb.map((b, i) =>
        `<span class="bc-item">${{b}}</span>` + (i < data.breadcrumb.length - 1 ? '<span class="bc-sep">›</span>' : '')
    ).join('');

    // Network info
    const netInfo = document.getElementById('network-info');
    const badgeData = [];
    if (data.network.n_nodes !== null) badgeData.push(['Nodes', data.network.n_nodes]);
    if (data.network.ratio_arc !== null) badgeData.push(['Arc Ratio', data.network.ratio_arc]);
    if (data.network.complexity_median) {{
        badgeData.push(['Complexity', `${{data.network.complexity_median.toFixed(0)}} [${{data.network.complexity_q1.toFixed(0)}}-${{data.network.complexity_q3.toFixed(0)}}]`]);
    }} else if (data.network.complexity) {{
        badgeData.push(['Complexity', data.network.complexity.toFixed(0)]);
    }}
    if (data.prior_types && data.prior_types.length > 0)
        badgeData.push(['Prior', data.prior_types[0]]);
    netInfo.innerHTML = badgeData.map(([k, v]) =>
        `<div class="net-badge"><span class="net-key">${{k}}</span><span class="net-val">${{v}}</span></div>`
    ).join('');

    buildControls(data);
    repositionExpandBtn();
    renderPlotly(data);
}}

function showEmptyState() {{
    emptyState.style.display = "flex";
    loading.classList.remove('visible');
    document.getElementById('breadcrumb').innerHTML = "";
    document.getElementById('network-info').innerHTML = "";
    document.getElementById('pop-buttons').innerHTML = "";
    document.getElementById('curve-buttons').innerHTML = "";
}}

function buildControls(data) {{
    const popWrap  = document.getElementById('pop-buttons');
    const mulLabel = document.getElementById('mul-label');
    const mulWrap  = document.getElementById('mul-buttons');

    if (viewMode === "multi_mul") {{
        // coverage singola, multiplier multipli
        popWrap.innerHTML = data.coverage_targets.map(pop =>
            `<button class="pop-btn ${{selectedPop === pop ? 'active' : ''}}"
             onclick="selectPopSingle(${{pop}})">${{pop}}</button>`
        ).join('');

        if (!window.selectedMulSet || window.selectedMulSet.size === 0)
            window.selectedMulSet = new Set(data.multipliers);

        mulLabel.style.display = '';
        mulWrap.innerHTML = data.multipliers.map(m =>
            `<button class="pop-btn ${{window.selectedMulSet.has(m) ? 'active' : ''}}"
            onclick="selectMul(${{m}})">${{m}}</button>`
        ).join('');

    }} else {{
        // multiplier singolo, coverage multiple
        if (!window.selectedMul || !data.multipliers.includes(window.selectedMul))
            window.selectedMul = data.multipliers[0];

        mulLabel.style.display = '';
        mulWrap.innerHTML = data.multipliers.map(m =>
            `<button class="pop-btn ${{window.selectedMul === m ? 'active' : ''}}"
             onclick="selectMulSingle(${{m}})">${{m}}</button>`
        ).join('');

        if (!window.selectedCovSet || window.selectedCovSet.size === 0)
            window.selectedCovSet = new Set(data.coverage_targets);

        popWrap.innerHTML = data.coverage_targets.map(pop =>
            `<button class="pop-btn ${{window.selectedCovSet.has(pop) ? 'active' : ''}}"
             onclick="selectCov(${{pop}})">${{pop}}</button>`
        ).join('');
    }}

    const curveWrap = document.getElementById('curve-buttons');
    curveWrap.innerHTML = data.methods.map(m => {{
        const isActive   = activeMethods.has(m.key);
        const isDisabled = !isActive && activeMethods.size >= {MAX_ACTIVE};
        return `<button class="curve-btn ${{isActive ? 'active' : ''}} ${{isDisabled ? 'disabled' : ''}}"
                    style="${{isActive ? 'color:' + m.color : ''}}"
                    ${{isDisabled ? 'disabled' : ''}}
                    onclick="toggleMethod('${{m.key}}')">
                <svg width="22" height="10" style="flex-shrink:0"><line x1="0" y1="5" x2="22" y2="5"
                    stroke="${{m.color}}" stroke-width="2.5"
                    stroke-dasharray="${{m.dash_css === 'none' ? '' : m.dash_css}}"/></svg>
                ${{m.label}}
            </button>`;
    }}).join('');
}}

function toggleMethod(methodKey) {{
    const data = plotsData[currentIdx];
    if (activeMethods.has(methodKey)) {{
        activeMethods.delete(methodKey);
    }} else {{
        if (activeMethods.size >= {MAX_ACTIVE}) return;
        activeMethods.add(methodKey);
    }}
    buildControls(data);
    renderPlotly(data);
}}

function selectPop(pop) {{
    if (selectedPopSet.has(pop)) {{
        if (selectedPopSet.size > 1) selectedPopSet.delete(pop); // keep at least one active
    }} else {{
        selectedPopSet.add(pop);
    }}
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

function renderPlotly(data) {{
    if (viewMode === "multi_mul") {{
        renderMultiMul(data); 
    }} else {{
        renderMultiCov(data);
    }}
}}

function renderMultiMul(data) {{
    const isDark = theme === "dark";
    const bg    = isDark ? "#181c27" : "#ffffff";
    const grid  = isDark ? "#2a3050" : "#cbd5e1";
    const text  = isDark ? "#e2e8f0" : "#0f172a";

    const multipliers = data.multipliers.filter(m => window.selectedMulSet && window.selectedMulSet.has(m));
    const nCols = multipliers.length;

    let layout = {{
        grid: {{ rows: 1, columns: nCols, pattern: 'coupled' }},
        plot_bgcolor: bg,
        paper_bgcolor: "rgba(0,0,0,0)",
        font: {{ color: text, family: "'Inter', sans-serif" }},
        margin: {{ l: 50, r: 20, t: 55, b: 70 }},
        showlegend: false,
        dragmode: 'pan',
        annotations: []
    }};

    const gap   = nCols > 1 ? 0.06 : 0.0;
    const slotW = (1 - gap * (nCols - 1)) / nCols;

    let filteredTraces = [];

    multipliers.forEach((mul, colIdx) => {{
        const colNum   = colIdx + 1;
        const xAxisKey = colIdx === 0 ? 'xaxis' : `xaxis${{colNum}}`;
        const yAxisKey = colIdx === 0 ? 'yaxis' : `yaxis${{colNum}}`;

        const xStart = colIdx * (slotW + gap);
        const xEnd   = xStart + slotW;

        layout[xAxisKey] = {{
            title: {{ text: 'False Positive Rate', font: {{ size: 12 }} }},
            range: [0, data.fpr_max],
            gridcolor: grid, linecolor: grid,
            zeroline: false,
            domain: [xStart, xEnd]
        }};
        layout[yAxisKey] = {{
            title: colIdx === 0
                ? {{ text: 'True Positive Rate', font: {{ size: 12 }} }}
                : null,
            range: [0, 1.02],
            gridcolor: grid, linecolor: grid,
            zeroline: false,
            showticklabels: colIdx === 0,
            anchor: colIdx === 0 ? 'x' : `x${{colNum}}`
        }};

        layout.annotations.push({{
            text: `Multiplier: ${{mul}}`,
            showarrow: false,
            x: 0.5, y: 1.08,
            xref: colIdx === 0 ? 'x domain' : `x${{colNum}} domain`,
            yref: 'paper', font: {{ size: 13, color: text }}
        }});

        data.traces.forEach(t => {{
            const meta = t.meta;
            const match =
                meta.multiplier === mul &&
                meta.pop        === selectedPop &&
                (meta.prior     === selectedPrior || meta.method_key === "__diag__");

            const methodActive =
                activeMethods.has(meta.method_key) ||
                meta.method_key === "__diag__";

            if (match && methodActive) {{
                let traceClone = JSON.parse(JSON.stringify(t));
                traceClone.visible = true;
                traceClone.xaxis = colIdx === 0 ? 'x' : `x${{colNum}}`;
                traceClone.yaxis = colIdx === 0 ? 'y' : `y${{colNum}}`;
                filteredTraces.push(traceClone);
            }}
        }});
    }});

    Plotly.react(plotDiv, filteredTraces, layout,
        {{ responsive: true, displayModeBar: true, displaylogo: false }})
        .then(() => {{
            loading.classList.remove('visible');
        }});
}}

function renderMultiCov(data) {{
    const isDark = theme === "dark";
    const bg   = isDark ? "#181c27" : "#ffffff";
    const grid = isDark ? "#2a3050" : "#cbd5e1";
    const text = isDark ? "#e2e8f0" : "#0f172a";

    const coverages = data.coverage_targets.filter(p => window.selectedCovSet && window.selectedCovSet.has(p));
    const nCols = coverages.length;
    const gap   = nCols > 1 ? 0.06 : 0.0;
    const slotW = (1 - gap * (nCols - 1)) / nCols;

    let layout = {{
        grid: {{ rows: 1, columns: nCols, pattern: 'coupled' }},
        plot_bgcolor: bg, paper_bgcolor: "rgba(0,0,0,0)",
        font: {{ color: text, family: "'Inter', sans-serif" }},
        margin: {{ l: 50, r: 20, t: 55, b: 70 }},
        showlegend: false, dragmode: 'pan', annotations: []
    }};

    let filteredTraces = [];

    coverages.forEach((pop, colIdx) => {{
        const colNum   = colIdx + 1;
        const xAxisKey = colIdx === 0 ? 'xaxis' : `xaxis${{colNum}}`;
        const yAxisKey = colIdx === 0 ? 'yaxis' : `yaxis${{colNum}}`;
        const xStart   = colIdx * (slotW + gap);

        layout[xAxisKey] = {{
            title: {{ text: 'False Positive Rate', font: {{ size: 12 }} }},
            range: [0, data.fpr_max], gridcolor: grid, linecolor: grid,
            zeroline: false, domain: [xStart, xStart + slotW]
        }};
        layout[yAxisKey] = {{
            title: colIdx === 0 ? {{ text: 'True Positive Rate', font: {{ size: 12 }} }} : null,
            range: [0, 1.02], gridcolor: grid, linecolor: grid,
            zeroline: false, showticklabels: colIdx === 0,
            anchor: colIdx === 0 ? 'x' : `x${{colNum}}`
        }};
        layout.annotations.push({{
            text: `Coverage: ${{pop}}`, showarrow: false,
            x: 0.5, y: 1.08,
            xref: colIdx === 0 ? 'x domain' : `x${{colNum}} domain`,
            yref: 'paper', font: {{ size: 13, color: text }}
        }});

        data.traces.forEach(t => {{
            const meta = t.meta;
            const match =
                meta.pop === pop &&
                meta.multiplier === window.selectedMul &&
                (meta.prior === selectedPrior || meta.method_key === "__diag__");
            const methodActive = activeMethods.has(meta.method_key) || meta.method_key === "__diag__";

            if (match && methodActive) {{
                let tc = JSON.parse(JSON.stringify(t));
                tc.visible = true;
                tc.xaxis = colIdx === 0 ? 'x' : `x${{colNum}}`;
                tc.yaxis = colIdx === 0 ? 'y' : `y${{colNum}}`;
                filteredTraces.push(tc);
            }}
        }});
    }});

    Plotly.react(plotDiv, filteredTraces, layout,
        {{ responsive: true, displayModeBar: true, displaylogo: false }})
        .then(() => {{ loading.classList.remove('visible'); }});
}}

// --- COLLAPSE PANELS INTERACTIVITY ---
function toggleSidebar() {{
    const sidebar = document.getElementById('sidebar');
    const expandBtn = document.getElementById('expand-btn');
    const colBtn = document.getElementById('collapse-btn');
    sidebar.classList.toggle('collapsed');
    const isCollapsed = sidebar.classList.contains('collapsed');
    document.documentElement.style.setProperty('--sidebar-w', isCollapsed ? '0px' : '280px');
    expandBtn.classList.toggle('visible', isCollapsed);
    colBtn.textContent = isCollapsed ? '▶' : '◀';
    setTimeout(() => {{ Plotly.Plots.resize(plotDiv); }}, 220);
}}

function toggleControls() {{
    const panel = document.getElementById('controls-panel');
    const arrow = document.getElementById('controls-toggle-arrow');
    const legend = document.getElementById('plot-legend');
    const expandBtn = document.getElementById('expand-btn');

    panel.classList.toggle('collapsed');
    arrow.classList.toggle('open');

    const isCollapsed = panel.classList.contains('collapsed');

    legend.classList.toggle('visible', isCollapsed);
    if (isCollapsed) buildStaticLegend();

    setTimeout(() => {{
        Plotly.Plots.resize(plotDiv);
        repositionExpandBtn();
    }}, 260);
}}

function buildStaticLegend() {{
    const data = plotsData[currentIdx];
    if (!data) return;
    const legend = document.getElementById('plot-legend');
    legend.innerHTML = data.methods
        .filter(m => activeMethods.has(m.key))
        .map(m => `
            <div class="legend-item">
                <svg width="22" height="10"><line x1="0" y1="5" x2="22" y2="5" stroke="${{m.color}}" stroke-width="2.5" stroke-dasharray="${{m.dash_css === 'none' ? '' : m.dash_css}}"/></svg>
                ${{m.label}}
            </div>
        `).join('');
}}

// --- RESIZE SIDEBAR LOGIC ---
function initResize() {{
    const handle = document.getElementById('resize-handle');
    const sidebar = document.getElementById('sidebar');
    let isDragging = false;
    
    handle.addEventListener('mousedown', (e) => {{
        isDragging = true;
        handle.classList.add('dragging');
        document.body.style.cursor = 'col-resize';
    }});
    
    document.addEventListener('mousemove', (e) => {{
        if (!isDragging) return;
        let newWidth = e.clientX;
        if (newWidth > 160 && newWidth < 520) {{
            sidebar.style.width = newWidth + 'px';
            document.documentElement.style.setProperty('--sidebar-w', newWidth + 'px');
        }}
    }});
    
    document.addEventListener('mouseup', () => {{
        if (!isDragging) return;
        isDragging = false;
        handle.classList.remove('dragging');
        document.body.style.cursor = 'default';
        Plotly.Plots.resize(plotDiv);
    }});
}}

// --- SEARCH FILTER ---
function onSearch() {{
    const query = document.getElementById('search').value.toLowerCase();
    
    document.querySelectorAll('.tree-leaf').forEach(leaf => {{
        leaf.classList.toggle('hidden', !leaf.textContent.toLowerCase().includes(query));
    }});
    
    document.querySelectorAll('.tree-details').forEach(details => {{
        const hasVisible = details.querySelectorAll('.tree-leaf:not(.hidden)').length > 0;
        details.style.display = hasVisible ? '' : 'none';
        if (query.length > 0 && hasVisible) details.open = true;
        if (query.length === 0) {{ details.style.display = ''; details.open = details.getAttribute('data-default-open') === 'true'; }}
    }});
}}

function repositionExpandBtn() {{
    const breadcrumb = document.getElementById('breadcrumb');
    const btn = document.getElementById('expand-btn');
    const rect = breadcrumb.getBoundingClientRect();
    btn.style.top = rect.top + 'px';
}}

function selectPrior(prior) {{
    selectedPrior = prior;
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

function selectPopSingle(pop) {{
    selectedPop = pop;
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

function expandToLeaf(idx) {{
    const leaf = document.querySelector(`.tree-leaf[data-idx="${{idx}}"]`);
    if (!leaf) return;
    let el = leaf.parentElement;
    while (el) {{
        if (el.tagName === 'DETAILS') el.open = true;
        el = el.parentElement;
    }}
}}

function selectMul(mul) {{
    if (window.selectedMulSet.has(mul)) {{
        if (window.selectedMulSet.size > 1) window.selectedMulSet.delete(mul);
    }} else {{
        window.selectedMulSet.add(mul);
    }}
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

function selectMulSingle(mul) {{
    window.selectedMul = mul;
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

function selectCov(pop) {{
    if (window.selectedCovSet.has(pop)) {{
        if (window.selectedCovSet.size > 1) window.selectedCovSet.delete(pop);
    }} else {{
        window.selectedCovSet.add(pop);
    }}
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

function toggleViewMode() {{
    viewMode = viewMode === "multi_mul" ? "multi_cov" : "multi_mul";
    const thumb = document.getElementById('toggle-thumb');
    const labelCov = document.getElementById('label-multicov');
    const labelMul = document.getElementById('label-multimul');
    if (viewMode === "multi_mul") {{
        thumb.style.transform = 'translateX(20px)';
        labelMul.style.color = 'var(--accent)';
        labelCov.style.color = 'var(--text3)';
    }} else {{
        thumb.style.transform = 'translateX(0px)';
        labelCov.style.color = 'var(--accent)';
        labelMul.style.color = 'var(--text3)';
    }}
    buildControls(plotsData[currentIdx]);
    renderPlotly(plotsData[currentIdx]);
}}

</script>
</body>
</html>"""

In [2]:
build_roc_report(
    root="./experiments/eval_unbalanced",          
    output="roc_report_eval_unbalanced.html",      # output file
    fpr_max=0.5,
    show_iqr=True,                 # IQR band
)

📂  Scanning experiments\eval_unbalanced …
✅  25 experiments found
   [ 25/25]  synthetic/simple_bn/exp2/exp5.3uake
🎨  Generating HTML …
✅  Report saved to: roc_report_eval_unbalanced.html  (26914 KB)


WindowsPath('roc_report_eval_unbalanced.html')

In [3]:
build_roc_report(
    root="./experiments/eval_balanced",          
    output="roc_report_eval_balanced.html",      # output file
    fpr_max=0.5,
    show_iqr=True,                 # IQR band
)

📂  Scanning experiments\eval_balanced …
✅  25 experiments found
   [ 25/25]  synthetic/simple_bn/exp2/exp5.3uake
🎨  Generating HTML …
✅  Report saved to: roc_report_eval_balanced.html  (26983 KB)


WindowsPath('roc_report_eval_balanced.html')